# ML Lab · Module 3, Unit 9: Forecasting inflation

**Prof. Dr. Mehmet Songur** · ML Lab, Module 3

Data: 13 monthly US series (FRED). The setup cell prepares a monthly table `G` (index: date). Columns: `inf` (annual inflation, %), `monthly` (monthly inflation, %), the annual change of 7 series (`CPILFESL`, `PPIACO`, `M2SL`, `WTISPLC`, `AHETPI`, `PAYEMS`, `INDPRO`), the level of 4 series (`UNRATE`, `FEDFUNDS`, `T10Y3MM`, `UMCSENT`) and `HOUST` (log). **Have Gemini write the code**, then **audit it**.

**Rules**
1. Do not change the **Setup** and **Check** cells; just run them.
2. Generate code with Gemini in the task cells; read it before running.
3. If a check cell passes, it gives you a **completion code**; paste it into the box in ML Lab.

> Note: there is no CPI for October 2025 (the US federal government shutdown). Don't fill missing months; leave out the rows that contain them. For simplicity we assume “this month's data are known at month end”; in reality CPI is published in the middle of the following month.

In [ ]:
#@title Setup — run it, do not change it

import io, base64, urllib.request, warnings
import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression, LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import TimeSeriesSplit, KFold
warnings.filterwarnings("ignore")

def fred_download(series_id):
    """Downloads a series from FRED; columns: date, <series_id>. FRED marks missing values with '.';
    they are deliberately turned into NaN and counted."""
    url = "https://fred.stlouisfed.org/graph/fredgraph.csv?id=" + series_id
    with urllib.request.urlopen(url) as response:
        df = pd.read_csv(io.BytesIO(response.read()))
    df.columns = ["date", series_id]
    df["date"] = pd.to_datetime(df["date"])
    df[series_id] = pd.to_numeric(df[series_id], errors="coerce")
    print(f"{series_id}: {len(df)} rows, {df[series_id].isna().sum()} missing, {df['date'].min():%Y-%m} – {df['date'].max():%Y-%m}")
    return df

codes = ['CPIAUCSL', 'CPILFESL', 'PPIACO', 'M2SL', 'WTISPLC', 'AHETPI', 'PAYEMS', 'INDPRO', 'UNRATE', 'FEDFUNDS', 'T10Y3MM', 'UMCSENT', 'HOUST']
raw = pd.concat([fred_download(k).set_index("date") for k in codes], axis=1).sort_index().asfreq("MS")
G = pd.DataFrame(index=raw.index)
G["inf"] = raw["CPIAUCSL"].pct_change(12, fill_method=None) * 100     # annual inflation (%)
G["monthly"] = raw["CPIAUCSL"].pct_change(1, fill_method=None) * 100  # monthly inflation (%)
for k in ["CPILFESL", "PPIACO", "M2SL", "WTISPLC", "AHETPI", "PAYEMS", "INDPRO"]:
    G[k] = raw[k].pct_change(12, fill_method=None) * 100              # annual change (%)
for k in ["UNRATE", "FEDFUNDS", "T10Y3MM", "UMCSENT"]:
    G[k] = raw[k]                                                      # level
G["HOUST"] = np.log(raw["HOUST"])                                      # housing starts (log)
_G = G.copy()
START, CUT = pd.Timestamp("1984-01-01"), pd.Timestamp("2015-01-01")

def _k(s):
    return base64.b64decode(s).decode()

def _reference():
    ft = {}
    for k in _G.columns:
        if k != "monthly":
            for g in range(3):
                ft[f"{k}_l{g}"] = _G[k].shift(g)
    for g in range(12):
        ft[f"monthly_l{g}"] = _G["monthly"].shift(g)
    X = pd.DataFrame(ft)
    t = pd.concat([X, _G["inf"].shift(-1).rename("target")], axis=1).loc[START:].dropna()
    e, s = t.loc[:CUT - pd.Timedelta(days=1)], t.loc[CUT:]
    c = list(X.columns)
    r = lambda a, b: float(np.sqrt(np.mean((np.asarray(a) - np.asarray(b)) ** 2)))
    sc = StandardScaler().fit(e[c])
    ols = LinearRegression().fit(sc.transform(e[c]), e["target"])
    las = LassoCV(alphas=np.logspace(-4, 0.5, 60), cv=TimeSeriesSplit(5), max_iter=50000).fit(sc.transform(e[c]), e["target"])
    return dict(n_test=len(s), naive=r(s["inf_l0"], s["target"]), ols=r(ols.predict(sc.transform(s[c])), s["target"]),
                lasso=r(las.predict(sc.transform(s[c])), s["target"]))

def check_A(table, train, test, rmse_naive, rmse_ols):
    if "target" not in table.columns:
        print("❌ No column called 'target' in 'table'."); return
    h = table["target"]
    if np.allclose(h.values, _G["inf"].reindex(h.index).values, equal_nan=True):
        print("❌ The target is not shifted: 'target' is this month's inflation. Next month's value needs shift(-1) (target leakage)."); return
    if not np.allclose(h.values, _G["inf"].shift(-1).reindex(h.index).values, equal_nan=True):
        print("❌ 'target' does not match next month's annual inflation (inf.shift(-1)). The target may be built differently or missing months may have been filled."); return
    if table.isna().sum().sum() > 0:
        print("❌ The table has missing values; rows with missing months should be left out (dropna)."); return
    if len(table.columns) - 1 != 51:
        print(f"❌ 51 features expected, found {len(table.columns) - 1}. Re-read the recipe: 13 series × 3 lags + 12 lags of monthly inflation."); return
    if not train.index.is_monotonic_increasing or not test.index.is_monotonic_increasing or train.index.max() >= CUT or test.index.min() < CUT:
        print("❌ The split is not by time: training before 2015, test 2015 onwards; no shuffling."); return
    r = _reference()
    if len(test) > r["n_test"]:
        print("❌ The test period has more months than expected: missing months may have been filled. Don't fill; leave them out."); return
    if train.index.min() > START + pd.Timedelta(days=40):
        print("❌ The training data do not start in 1984."); return
    if abs(float(rmse_naive) - r["naive"]) > 0.005:
        print("❌ The naive forecast's RMSE differs from the expected value. Is the forecast 'this month's annual inflation' (inf_l0), compared with the target?"); return
    if abs(float(rmse_ols) - r["ols"]) > 0.02:
        print("❌ OLS's test RMSE differs from the expected value. Were all 51 features used? Was the model built on training data only?"); return
    print(f"✅ All checks passed (naive ≈ {r['naive']:.3f}, OLS ≈ {r['ols']:.3f}). Your completion code: " + _k("TUwzLUVORi00S1Fa"))

def check_B(lasso, rmse_lasso, selected):
    if isinstance(getattr(lasso, "cv", None), KFold) or not isinstance(getattr(lasso, "cv", None), TimeSeriesSplit):
        print("❌ λ was not chosen with time-respecting cross-validation. Use cv=TimeSeriesSplit(5); shuffled KFold shows the future."); return
    r = _reference()
    if abs(float(rmse_lasso) - r["lasso"]) > 0.01:
        print("❌ Lasso's test RMSE differs from the expected value. Was the scaler fitted on training data only? Do the alphas grid and max_iter match the prompt?"); return
    nz = int(np.sum(np.abs(lasso.coef_) > 0))
    if len(selected) != nz:
        print(f"❌ 'selected' has {len(selected)} items, but there are {nz} non-zero coefficients. Count negative coefficients too (coef_ != 0)."); return
    if "monthly_l11" not in selected:
        print("ℹ️ Note: monthly_l11 (base effect) is not on the list; check the feature names.")
    print(f"✅ All checks passed (Lasso ≈ {r['lasso']:.3f}, naive ≈ {r['naive']:.3f}). Your completion code: " + _k("TUwzLUxTTy03WFBE"))

print("G table:", G.shape)
G.tail(3).round(2)

## Task A — Problem and benchmark

1. **Features:** for every column of `G` except `monthly`, its values lagged 0, 1 and 2 months (`{column}_l0`, `{column}_l1`, `{column}_l2`), and `monthly` lagged 0–11 months (`monthly_l0` … `monthly_l11`): 51 in total.
2. **Target:** next month's annual inflation, a column called `target`: `G["inf"].shift(-1)`.
3. Combine the features and the target in a table called `table`, start it at `1984-01-01` and drop rows with missing values.
4. **Split by time:** before `2015-01-01` → `train`, from then on → `test`. Put the feature names in a `features` list.
5. Over the test period, compute the RMSE of the **naive forecast** (`inf_l0`) and of a **51-variable OLS** standardised and fitted on the training data: `rmse_naive`, `rmse_ols`.

**Audit questions:** Is the target `shift(-1)`? Any negative shifts in the features? Any shuffling in the split? Were missing months filled?

In [ ]:
# Task A: generate the code with Gemini here, read it, then run it.

### Check A — run it, do not change it

In [ ]:
check_A(table, train, test, rmse_naive, rmse_ols)

## Task B — Lasso

1. Fit `StandardScaler` on `train[features]` **only**; apply only `transform` to the test data.
2. Choose λ with `LassoCV(alphas=np.logspace(-4, 0.5, 60), cv=TimeSeriesSplit(5), max_iter=50000)`; store the model in `lasso`.
3. Store the test RMSE in `rmse_lasso` and compare it with the naive forecast.
4. Put the features with non-zero coefficients in a `selected` list; print the top 8 by absolute value with their coefficients.

**Audit questions:** What was the scaler fitted on? What is `cv`? What do the selected variables say economically (base effect: `monthly_l11`)?

In [ ]:
# Task B: generate the code with Gemini here, read it, then run it.

### Check B — run it, do not change it

In [ ]:
check_B(lasso, rmse_lasso, selected)

## Your audit notes

- How did Gemini build the target and the split? Did you need to change it?
- How were scaling and cross-validation done?
- By how much does Lasso beat the naive forecast? Which variables were selected, and what do they mean?